In [1]:
import sys # for automation and parallelization: set manual to false when run by a launcher
import json
MANUAL, SCENARIO = (True, 'base') if 'ipykernel' in sys.argv[0]\
     else (False,  json.loads(sys.argv[1])['scenario'])

# START

In [2]:
sys.path.insert(0, r'../../../quetzal')

from quetzal.model import connectionscanmodel, stepmodel
from quetzal.io import excel
import numpy as np
import pandas as pd
import geopandas as gpd

import shapely

import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
qpath = r'../../'
ipath = qpath + r'inputs/'
mpath = qpath + r'scenarios/{}/models/'.format(SCENARIO)

In [4]:
var = excel.read_var(ipath + r'parameters.xlsx', scenario=SCENARIO)

Index(['description', 'unit', 'category', 'parameter', 'base',
       'base_timetables'],
      dtype='object')


In [5]:
#if var['run_mode']['mode'] == 'timetables':
#    print('timetables scenario')
#    end_of_notebook

In [ ]:
NTLEG_PENALTY = var[('pathfinders_general', 'ntleg_penalty')]
TRANSFER_DURATION = var[('pathfinders_general', 'transfer_duration')]
WALK_SPEED = var[('pathfinders_general', 'walk_speed')]
BIKE_SPEED = var[('pathfinders_general', 'bike_speed')]
TRANSFER_PENALTY = var[('pathfinders_general', 'transfer_penalty')]

WALK_TIME_FACTOR = var[('mode_penalty', 'walk_time_factor')]
BIKE_TIME_FACTOR = var[('mode_penalty', 'bike_time_factor')]
CAR_TIME_FACTOR = var[('mode_penalty', 'car_time_factor')]
BOARDING_TIMES = var['boarding_time'].to_dict()

FLAT_FARES = var['fare'].to_dict()
KM_FARES = var['fare_km'].to_dict()

CAR_KM_FARES = var['fare_car'].to_dict()
KM_FARES['car'] = CAR_KM_FARES['fuel_cost'] + CAR_KM_FARES['other_costs']

VALUE_OF_TIME_SEC = var[('value_of_time', 'value_of_time_h')]/3600

INTRA_AREA_TO_DISTANCE_FACTOR = var[('step_distribution', 'intra_area_to_distance_factor')]

# READ

In [7]:
sm = stepmodel.read_zippedpickles(mpath + 'zones')

zones: 100%|██████████| 13/13 [00:00<00:00, 30.27it/s]     


# NODE CLUSTERING

In [8]:
sm.preparation_clusterize_nodes(
     distance_threshold=200
)

In [9]:
sm._add_type_prefixes(
        prefixes={'nodes': 'node_'}
    )

can not add prefixes on table:  footpaths


# Walkmodels

In [10]:
sm.road_links['walk_speed'] = WALK_SPEED # km / h
sm.road_links['walk_time'] = sm.road_links.length / (sm.road_links['walk_speed'] / 3.6) * WALK_TIME_FACTOR

In [11]:
sm.preparation_ntlegs(
    zone_to_road=True,
    zone_to_transit=False,
    short_leg_speed=WALK_SPEED/WALK_TIME_FACTOR,
    long_leg_speed=WALK_SPEED/WALK_TIME_FACTOR
)

In [12]:
sm._add_type_prefixes(
    prefixes={
        'zone_to_road': 'ztr_',
        'road_to_transit': 'rtt_',
    }
)

# PATHFINDER - PT ONLY

In [13]:
# Access times
sm.links['time'] += sm.links['route_type'].map(BOARDING_TIMES).fillna(0)

In [14]:
# add fare in time for pathfinder to find most relevant paths
sm.links['km_fare'] = sm.links.length * sm.links.route_type.apply(lambda x: KM_FARES[x]) / 1000
sm.links['true_time'] = sm.links['time'].copy()
sm.links['time'] = sm.links['time'] + (sm.links['km_fare'] / VALUE_OF_TIME_SEC)

In [15]:
# pruning algorithm
sm.step_pt_pathfinder(
    broken_modes=True, broken_routes=True,
    walk_on_road=True, keep_pathfinder=True
)

# remove intrazonal (handled differently)
sm.pt_los = sm.pt_los[sm.pt_los['origin']!=sm.pt_los['destination']]
sm.pt_los.reset_index(drop=True, inplace=True)

start publicpathfinder
68 sources 68 targets direct search
{'route_type': [{'rail'}, {'tramway'}, {'rail', 'tramway'}, {'bus'}, {'rail', 'bus'}, {'tramway', 'bus'}, {'rail', 'tramway', 'bus'}]}
68 + 0 = 68 >= 0.5 * 68 keeping od_set
3 + 9 = 12 < 0.5 * 68 splitting od_set
7 + 68 = 75 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
8 + 68 = 76 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
68 + 0 = 68 >= 0.5 * 68 keeping od_set
2 + 1 = 3 >= 0.5 * 4 keeping od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_set
6 + 7 = 13 < 0.5 * 33 splitting od_set
0 + 1 = 1 >= 0.5 * 1 keeping od_set
7 + 3 = 10 < 0.5 * 66 splitting od_set
16 + 8 = 24 < 0.5 * 57 splitting od_set
1 + 1 = 2 < 0.5 * 41 splitting od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_set
3 + 4 = 7 < 0.5 * 63 splitting od_set
0 + 3 = 3 >= 0.5 * 3 keeping od_set
2 + 0 = 2 >= 0.5 * 2 keeping od_set
1 + 0 = 1 >= 0.5 * 1 keeping od_set
6 + 1 = 7 >= 0.5 * 7 keeping od_set
1 + 0 = 1 >= 0.

KeyboardInterrupt: 

In [ ]:
# pruning algorithm
sm.step_pt_pathfinder(
    broken_modes=False, broken_routes=False,
    walk_on_road=True, keep_pathfinder=True
)

# remove intrazonal (handled differently)
sm.pt_los = sm.pt_los[sm.pt_los['origin']!=sm.pt_los['destination']]
sm.pt_los.reset_index(drop=True, inplace=True)

In [ ]:
hierarchy = ['rail', 'tramway', 'bus', 'schoolbus', 'ferry', 'walk']
sm.analysis_pt_route_type(hierarchy)

In [ ]:
# COMPUTE FARES
links_mode = {}
links_mode_lengths = {}
for mode in sm.links.route_type.unique():
    links = sm.links[sm.links.route_type==mode].index
    links_mode[mode] = links
    lengths = sm.links.loc[links]['length'].to_dict()
    links_mode_lengths[mode] = lengths

for k,v in links_mode.items():
    sm.pt_los[k + '_link_path'] = sm.pt_los['link_path'].apply(lambda x: [a for a in x if a in v])
    sm.pt_los[k + '_in_vehicle_length'] = sm.pt_los[k + '_link_path'].apply(lambda x: sum([links_mode_lengths[k][a] for a in x]))

sm.pt_los['fare'] = 0
sm.pt_los['flat_fare'] = 0
sm.pt_los['km_fare'] = 0

for m in links_mode.keys():
    sm.pt_los['fare'] += (sm.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m] +\
          sm.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000
    sm.pt_los['flat_fare'] += (sm.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m]
    sm.pt_los['km_fare'] += sm.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000

sm.pt_los['parking_fare'] = 0

# PATHFINDER - PT + BIKE

In [ ]:
sm_velo = sm.copy()

In [ ]:
sm_velo.road_links = sm_velo.road_links.loc[sm_velo.road_links['bike']]
sm_velo.road_links['walk_speed'] = BIKE_SPEED
sm_velo.road_links['walk_time'] = sm_velo.road_links.length / (sm_velo.road_links['walk_speed'] / 3.6) * BIKE_TIME_FACTOR

sm_velo.integrity_fix_road_network()

In [ ]:
sm_velo.preparation_ntlegs(
    zone_to_road=True,
    zone_to_transit=False,
    short_leg_speed=BIKE_SPEED/BIKE_TIME_FACTOR,
    long_leg_speed=BIKE_SPEED/BIKE_TIME_FACTOR
)
sm_velo._add_type_prefixes(
    prefixes={
        'zone_to_road': 'ztr_',
        'road_to_transit': 'rtt_',
    }
)

In [ ]:
sm_velo.links = sm_velo.links.loc[sm_velo.links['route_type'].isin(['rail', 'tramway'])]
sm_velo.nodes = sm_velo.nodes.loc[list(set(sm_velo.links['a'].values.tolist() + sm_velo.links['b'].values.tolist()))]

# pruning algorithm
sm_velo.step_pt_pathfinder(
    broken_modes=True, broken_routes=False,
    walk_on_road=True
)

# remove intrazonal (handled differently)
sm_velo.pt_los = sm_velo.pt_los[sm_velo.pt_los['origin']!=sm_velo.pt_los['destination']]
sm_velo.pt_los.reset_index(drop=True, inplace=True)

In [ ]:
hierarchy = ['rail', 'tramway', 'bike']
sm_velo.analysis_pt_route_type(hierarchy)

sm_velo.pt_los['route_types'] = sm_velo.pt_los['route_types'].apply(lambda x: x+('bike',))

In [ ]:
# COMPUTE FARES
links_mode = {}
links_mode_lengths = {}
for mode in sm_velo.links.route_type.unique():
    links = sm_velo.links[sm_velo.links.route_type==mode].index
    links_mode[mode] = links
    lengths = sm_velo.links.loc[links]['length'].to_dict()
    links_mode_lengths[mode] = lengths

for k,v in links_mode.items():
    sm_velo.pt_los[k + '_link_path'] = sm_velo.pt_los['link_path'].apply(lambda x: [a for a in x if a in v])
    sm_velo.pt_los[k + '_in_vehicle_length'] = sm_velo.pt_los[k + '_link_path'].apply(lambda x: sum([links_mode_lengths[k][a] for a in x]))

sm_velo.pt_los['fare'] = 0
sm_velo.pt_los['km_fare'] = 0
sm_velo.pt_los['flat_fare'] = 0

for m in links_mode.keys():
    sm_velo.pt_los['fare'] += (sm_velo.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m] +\
          sm_velo.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000
    sm_velo.pt_los['flat_fare'] += (sm_velo.pt_los[m + '_in_vehicle_length']>0) * FLAT_FARES[m]
    sm_velo.pt_los['km_fare'] += sm_velo.pt_los[m + '_in_vehicle_length'] * KM_FARES[m] / 1000

sm_velo.pt_los['parking_fare'] = 0

# Correct times

In [ ]:
# TC
sm.links['pathfinder_time'] = sm.links['time']
sm.links['time'] = sm.links['true_time']
sm.road_links['walk_time'] *= 1 / WALK_TIME_FACTOR
sm.zone_to_road['time'] *= 1 / WALK_TIME_FACTOR
sm.analysis_pt_time(boarding_time=0, walk_on_road=True)

# Vélo
sm_velo.links['pathfinder_time'] = sm_velo.links['time']
sm_velo.links['time'] = sm_velo.links['true_time']
sm_velo.zone_to_road['time'] *= 1 / BIKE_TIME_FACTOR
sm_velo.analysis_pt_time(boarding_time=0, walk_on_road=True)

In [ ]:
sm.pt_los = pd.concat([sm.pt_los, sm_velo.pt_los])

# ROAD PATHFINDER | car

In [ ]:
# empty volumes
sm.volumes = pd.DataFrame(index=pd.MultiIndex.from_product([sm.zones.index]*2))
sm.volumes.index.names = ['origin', 'destination']
sm.volumes.reset_index(inplace=True)

In [ ]:
sm.step_road_pathfinder(method='aon')
# remove intrazonal (handled differently)
sm.car_los = sm.car_los[sm.car_los['origin']!=sm.car_los['destination']]
sm.car_los['pathfinder_session'] = 'car_aon'

In [ ]:
sm.analysis_car_los()
sm.analysis_car_length()

# ROAD PATHFINDER | bike

In [ ]:
sm_road_bike = sm_velo.copy()
### No bikes on motorways
# TODO: remplacer ça par un attribut "bike" à False dans le réseau routier et 
# mettre cette préparation dans le script networks (integrity_fix_road_network sert à modifier l’attribut bike),
# et utiliser ce réseau également pour les footpaths précédents (vélo+TC)
### sm_road_bike.road_links = sm_road_bike.road_links.loc[~sm_road_bike.road_links['highway'].isin(['motorway', 'motorway_link', 'trunk', 'trunk_link'])]
### sm_road_bike.road_nodes = sm_road_bike.road_nodes.loc[sm_road_bike.road_nodes.index.isin(sm_road_bike.road_links['a'].values.tolist()) | sm_road_bike.road_nodes.index.isin(sm_road_bike.road_links['b'].values.tolist())]

### sm_road_bike.integrity_fix_road_network()
### 
### sm_road_bike.preparation_ntlegs(
###     zone_to_road=True,
###     zone_to_transit=False,
###     short_leg_speed=BIKE_SPEED/BIKE_TIME_FACTOR,
###     long_leg_speed=BIKE_SPEED/BIKE_TIME_FACTOR
### )
### sm_road_bike._add_type_prefixes(
###     prefixes={
###         'zone_to_road': 'ztr_',
###         'road_to_transit': 'rtt_',
###     }
### )
### 
sm_road_bike.road_links['speed'] = BIKE_SPEED
sm_road_bike.road_links['time'] = sm_road_bike.road_links.length / (sm_road_bike.road_links['speed'] / 3.6) * BIKE_TIME_FACTOR

In [ ]:
# empty volumes
sm_road_bike.volumes = pd.DataFrame(index=pd.MultiIndex.from_product([sm.zones.index]*2))
sm_road_bike.volumes.index.names = ['origin', 'destination']
sm_road_bike.volumes.reset_index(inplace=True)

In [ ]:
sm_road_bike.step_road_pathfinder(method='aon')
# remove intrazonal (handled differently)
sm_road_bike.car_los = sm_road_bike.car_los[sm_road_bike.car_los['origin']!=sm_road_bike.car_los['destination']]
sm_road_bike.car_los['pathfinder_session'] = 'bike_aon'

In [ ]:
sm_road_bike.analysis_car_los()
sm_road_bike.analysis_car_length()

# INTRAZONAL

In [ ]:
sm.zone_length = (sm.zones.area ** 0.5) * INTRA_AREA_TO_DISTANCE_FACTOR

In [ ]:
def set_intra_los(mode, speed_kmh, km_fare, base_fare=0, time_penalty=0):
    intra_los = sm.volumes[sm.volumes['origin']==sm.volumes['destination']].copy()

    intra_los['route_type'] = mode
    if mode != 'walk':
        intra_los['route_types'] = [(mode,)]*len(intra_los)
    else:
        intra_los['route_types'] = [tuple()]*len(intra_los)

    intra_los['length'] = intra_los['origin'].apply(lambda x: sm.zone_length[x])
    intra_los['gtime'] = intra_los['origin'].apply(
        lambda x: sm.zone_length[x] / (speed_kmh / 3.6)
    )
    intra_los['gtime'] += time_penalty
    intra_los['time'] = intra_los['gtime']
    intra_los['fare'] = intra_los['origin'].apply(
        lambda x: sm.zone_length[x] * km_fare / 1000 + base_fare
    )
    intra_los['pathfinder_session'] = 'intrazonal'
    intra_los['ntransfers'] = 0
    intra_los['path'] = intra_los['origin'].apply(lambda x: tuple([x]))

    for col in ['node_path', 'link_path', 'boardings', 'alightings',
        'footpaths', 'ntlegs', 'boarding_links', 'alighting_links', 'transfers']:
            intra_los[col] = [tuple()]*len(intra_los)

    return intra_los

sm.intra_los = pd.concat(
    [
        set_intra_los('car', 40/CAR_TIME_FACTOR, KM_FARES['car'], base_fare=0),
        set_intra_los('bike', BIKE_SPEED/BIKE_TIME_FACTOR, 0),
        set_intra_los('walk', WALK_SPEED/WALK_TIME_FACTOR, 0)
    ]
)

sm.intra_los.reset_index(drop=True, inplace=True)

In [ ]:
sm.intra_los['flat_fare'] = 0
sm.intra_los['km_fare'] = sm.intra_los['fare'].copy()

sm.intra_los['parking_fare'] = 0
sm.intra_los.loc[sm.intra_los['route_type'] == 'car', 'parking_fare'] = sm.intra_los.loc[sm.intra_los['route_type'] == 'car', 'destination'].map(sm.zones['parking_fare'].to_dict())
sm.intra_los['fare'] += sm.intra_los['parking_fare']

# CONCAT

In [ ]:
# CAR
sm.car_los['penalties'] = 0
sm.car_los['ntransfers'] = 0
sm.car_los['parking_fare'] = sm.car_los['destination'].map(sm.zones['parking_fare'].to_dict())
sm.car_los['flat_fare'] = 0
sm.car_los['km_fare'] = sm.car_los['length'] * KM_FARES['car'] / 1000 
sm.car_los['fare'] = sm.car_los['length'] * KM_FARES['car'] / 1000 + sm.car_los['parking_fare']
sm.car_los['gtime'] *= CAR_TIME_FACTOR
sm.car_los['gtime'] += sm.car_los['penalties']
sm.car_los['route_types'] = [('car',)] * len(sm.car_los)
sm.car_los['route_type'] = 'car'
sm.car_los['time'] = sm.car_los['gtime']
sm.car_los['gtime'] += sm.car_los['in_vehicle_length'] / 1000 * KM_FARES['car'] / VALUE_OF_TIME_SEC

# BIKE
sm_road_bike.car_los['penalties'] = 0
sm_road_bike.car_los['ntransfers'] = 0
sm_road_bike.car_los['parking_fare'] = 0
sm_road_bike.car_los['flat_fare'] = 0
sm_road_bike.car_los['km_fare'] = 0
sm_road_bike.car_los['fare'] = 0
sm_road_bike.car_los['gtime'] *= BIKE_TIME_FACTOR
sm_road_bike.car_los['gtime'] += sm_road_bike.car_los['penalties']
sm_road_bike.car_los['route_types'] = [('bike',)] * len(sm.car_los)
sm_road_bike.car_los['route_type'] = 'bike'
sm_road_bike.car_los['time'] = sm.car_los['gtime']

sm.los = pd.concat([sm.car_los, sm_road_bike.car_los, sm.pt_los, sm.intra_los]).reset_index(drop=True)

# EXPORT

In [ ]:
print(sm.los['route_types'].unique())

sm.to_zippedpickles(
    mpath + r'pathfinders',
    omitted_attributes=['pt_los', 'car_los', 'intra_los']
)

In [ ]:
end_of_notebook